In [9]:
from gensim.models import Word2Vec
import os
import json
import pandas as pd
import numpy as np
from CADA.paths import DATA_DIRECTORY
import pickle
from CADA.disease_gene_mapping import disease_gene


In [10]:
model = Word2Vec.load('/home/peng/PycharmProjects/CADA/model/all_patients_disease_doublefalse/with_patients.model')

In [11]:
with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'gene_id_name.dict'), 'rb') as handle:
    gene_id_name = pickle.load(handle)

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'gene_name_id.dict'), 'rb') as handle:
    gene_name_id = pickle.load(handle)

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'hpo_id_name.dict'), 'rb') as handle:
    hpo_id_name = pickle.load(handle)

dict_disease_gene = disease_gene()

In [16]:
input_dir = os.path.join(DATA_DIRECTORY, 'raw', 'patients', 'evaluation_from_jonathanz', 'cases_new.json')
results_tsv = []
results_excel = []

with open(input_dir) as infile:
    content = json.load(infile)
    for patient in content:
        features = []
        case = 'Patient:' + str(patient['case'])
        gene_name = patient['gt_gene'][0]                                                                                                                       
        gene_id = gene_name_id[gene_name]
        for feature in patient['hpo_num']:
            features.append('HP:' + str(feature.zfill(7)))
        if len(features) > 0:
            similar_nodes = model.most_similar(positive=features, topn=2000000)
            disease_nodes = []
            gene_nodes = []
            for node in similar_nodes:
                if node[0].startswith('OMIM:'):
                    disease_nodes.append(node[0])
            for disease in disease_nodes:
                if disease in dict_disease_gene:
                        gene_list = dict_disease_gene[disease]
                        gene_nodes += gene_list
        
            rank = gene_nodes.index(gene_id) + 1
                        
        results_tsv.append([case, ','.join(features), gene_id,','.join(gene_nodes[:30]), rank])
        results_excel.append([case, ','.join([hpo_id_name[feature] for feature in features]), gene_name, ','.join(gene_id_name[id] for id in gene_nodes[:30]), rank])
    
    df_tsv = pd.DataFrame(results_tsv, columns = ['Case_id', 'HPO_ids', 'Gene','CADA_top30', 'CADA_rank'])
    df_tsv.to_csv('result_from_disease.tsv', sep='\t', index=None)
    df_excel = pd.DataFrame(results_excel, columns = ['Case_id', 'HPO_terms', 'Gene', 'CADA_top30', 'CADA_rank'])
    df_excel.to_excel('result_from_disease.xlsx', index=None)

/home/peng/.local/lib/python3.7/site-packages/ipykernel_launcher.py:15: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  from ipykernel import kernelapp as app
